# Wrong labels: does the ranking change?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adeshinan/fitrank/blob/main/examples/label_noise_demo.ipynb)
[![PyPI](https://img.shields.io/pypi/v/fitrank)](https://pypi.org/project/fitrank/)
[![Docs](https://img.shields.io/badge/docs-EN%20%7C%20FR-blue)](https://adeshinan.github.io/fitrank/)

Real datasets contain wrong labels. When 20 % of the training labels are wrong, does the **ranking of the models** change? And if we remove the rows that look wrong, does it come back?

This notebook answers on one dataset with the tools added in FitRank 1.2:

1. split the data **once**, and keep a clean test set apart (`test_data`);
2. add 20 % wrong labels to the training rows;
3. rank the models on clean and noisy data, and **compare the two rankings** (`compare_rankings`);
4. find the suspicious rows with **out-of-fold probabilities** (`oof_proba`) from a **calibrated** model (`calibrate`), and remove them;
5. rank again on the cleaned data, and compare with the clean ranking.

It runs in about a minute. It is an example of how to use FitRank, with noise added on purpose: one dataset and one seed are not a study.

In [ ]:
%pip install -q "fitrank>=1.2"
import fitrank

print("FitRank", fitrank.__version__)

## 1. The data, split once

The [Dry Bean dataset](https://archive.ics.uci.edu/dataset/602/dry+bean+dataset) (UCI) describes 13,611 beans of 7 varieties by 16 shape measures (area, perimeter, roundness...). We keep 5,000 beans to stay fast, then put 30 % aside as the test set. **The test set stays clean** and is the same for every run below: only the training rows change.

In [ ]:
import http.client
import io
import time
import urllib.request
import zipfile

import pandas as pd
from scipy.io import arff
from sklearn.model_selection import train_test_split

URL = "https://archive.ics.uci.edu/static/public/602/dry+bean+dataset.zip"


def download(url, attempts=10):
    """The UCI server sometimes cuts a download short: try again."""
    for attempt in range(1, attempts + 1):
        try:
            with urllib.request.urlopen(url, timeout=60) as response:
                return response.read()
        except (OSError, ValueError, http.client.HTTPException):
            if attempt == attempts:
                raise
            time.sleep(1)


archive = zipfile.ZipFile(io.BytesIO(download(URL)))
text = archive.read("DryBeanDataset/Dry_Bean_Dataset.arff").decode("utf-8")
beans = pd.DataFrame(arff.loadarff(io.StringIO(text))[0])
beans["Class"] = beans["Class"].str.decode("utf-8")
beans = beans.groupby("Class").sample(frac=5000 / len(beans), random_state=0)

train, test = train_test_split(beans, test_size=0.3, stratify=beans["Class"], random_state=0)
print(f"{len(train)} training rows, {len(test)} test rows")
beans["Class"].value_counts()

## 2. Add 20 % wrong labels

We pick 20 % of the training rows at random and give each one another variety, chosen at random (*uniform* noise). A few lines of numpy: FitRank never changes your data.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
train_noisy = train.copy()
flipped = rng.choice(train_noisy.index, size=int(0.2 * len(train_noisy)), replace=False)
varieties = np.sort(beans["Class"].unique())
for row in flipped:
    others = varieties[varieties != train_noisy.loc[row, "Class"]]
    train_noisy.loc[row, "Class"] = rng.choice(others)

print(f"{len(flipped)} wrong labels out of {len(train_noisy)} training rows")

## 3. Rank on clean and noisy data

Both runs use the same models, the same folds and the same metric, and are scored on the same clean test set (`test_data`).

`svm` has no class probabilities. `calibrate: sigmoid` adds a calibrated copy of it, `svm (calibrated: sigmoid)`, which has them: we will need them in step 4.

In [ ]:
from fitrank import Benchmark

features = [column for column in beans.columns if column != "Class"]
setup = {
    "target": "Class",
    "task": "classification",
    "preprocessing": {"numeric": {"columns": features, "steps": ["scaler"]}},
    "models": [
        {"name": "logistic_regression", "params": {"C": [1]}},
        {"name": "knn", "params": {"n_neighbors": [15]}},
        {"name": "decision_tree", "params": {"max_depth": [8]}},
        {"name": "random_forest", "params": {"n_estimators": [100]}},
        {"name": "svm", "params": {"C": [1]}},
        {"name": "svm", "params": {"C": [1]}, "calibrate": "sigmoid"},
    ],
    "evaluation": {"cv": 3, "primary_metric": "f1_macro"},
}
cols = ["rank", "model", "cv_f1_macro", "test_f1_macro"]

clean = Benchmark(data=train, test_data=test, **setup).run(export=False)
noisy = Benchmark(data=train_noisy, test_data=test, **setup).run(export=False)

clean.leaderboard()[cols].round(3)

In [ ]:
noisy.leaderboard()[cols].round(3)

The cross-validation scores drop a lot: they are computed on noisy labels too. The test scores, on clean labels, drop much less. But **did the order change?** `compare_rankings` answers in three numbers:

- `kendall_tau`: 1 = the same order, 0 = unrelated, -1 = reversed;
- `top1_agree`: is the best model the same on both sides?
- `regret`: the test score lost, on the clean side, by taking the noisy winner.

In [ ]:
from fitrank import compare_rankings

before = compare_rankings(clean, noisy)
print(f"tau = {before.kendall_tau:.2f}, same best model: {before.top1_agree}")
print(f"regret = {before.regret:.4f}")
before.table[["model", "reference_rank", "other_rank", "rank_shift"]]

## 4. Find the suspicious rows

`oof_proba` gives, for every training row, the probabilities predicted by a model **that never saw that row** (the other folds of the cross-validation). A row whose label gets a very low probability is suspicious.

`svm` cannot give probabilities, and FitRank says how to fix it:

In [ ]:
from fitrank import FitRankError

try:
    noisy.oof_proba("svm")
except FitRankError as error:
    print(error)

The calibrated copy has them. We compute how much the model believes each observed label, and remove the 20 % of rows it believes least (the *low-confidence* rule).

In [ ]:
proba = noisy.oof_proba("svm (calibrated: sigmoid)")  # one row per training row
labels = noisy.y_train.loc[proba.index]  # the observed (noisy) labels
confidence = proba.to_numpy()[range(len(proba)), proba.columns.get_indexer(labels)]

suspects = proba.index[np.argsort(confidence)[: len(flipped)]]  # the 20 % least believed
train_cleaned = train_noisy.drop(index=suspects)

# Only possible here because we added the noise ourselves:
print(f"{np.isin(suspects, flipped).sum()} of the {len(suspects)} removed rows had a wrong label")
proba.head().round(3)

## 5. Rank again on the cleaned data

Same setup, same clean test set. Is the cleaned ranking closer to the clean one than the noisy ranking was?

In [ ]:
cleaned = Benchmark(data=train_cleaned, test_data=test, **setup).run(export=False)
after = compare_rankings(clean, cleaned)

pd.DataFrame(
    {
        "kendall_tau": [before.kendall_tau, after.kendall_tau],
        "top1_agree": [before.top1_agree, after.top1_agree],
        "regret": [before.regret, after.regret],
    },
    index=["clean vs noisy", "clean vs cleaned"],
).round(3)

In [ ]:
cleaned.leaderboard()[cols].round(3)

## Before drawing conclusions

- **One dataset, one seed.** With 6 models, tau takes few values and moves from one `random_state` to another. Run the same comparison with several seeds, and compute the tau between two seeds on the same data: that is the size of the noise.
- **Removing rows also removes good ones.** Compare the number of removed rows with the number of wrong labels, and try other thresholds.
- **The probabilities are not fully out-of-sample**: the best parameters were chosen with the same folds. `oof_proba(..., params="default")` is stricter.
- **Calibration**: `sigmoid` suits small datasets; `isotonic` is more flexible but can overfit below about 1,000 rows. `neg_brier_score` in `evaluation.metrics` measures it.

## Next steps

- 📖 **Documentation** (English and French): https://adeshinan.github.io/fitrank/, with the guide "Measure the effect of a data cleaning"
- 🧪 **The first notebook**, to discover FitRank: [try_fitrank.ipynb](https://colab.research.google.com/github/Adeshinan/fitrank/blob/main/examples/try_fitrank.ipynb)
- 💬 **Feedback, bugs, ideas**: https://github.com/Adeshinan/fitrank/issues

FitRank is open source (MIT). Dry Bean dataset: Koklu, M. and Ozkan, I.A. (2020), *Multiclass Classification of Dry Beans Using Computer Vision and Machine Learning Techniques*, Computers and Electronics in Agriculture, 174, 105507 (CC BY 4.0).